# 05 - Delta Optimization

## Purpose

Inspect and optimize Gold Delta tables after the model has been created and validated.

## Concepts covered

- Delta table health inspection
- Row counts and table details
- OPTIMIZE for file compaction
- Safe VACUUM guidance
- Query timing for before and after comparison
- Operational recommendations for Fabric Lakehouse tables

## Prerequisites

03_gold_dimensional_model.ipynb completed successfully and Gold Delta tables exist.

## Expected output

Table health summary, optional optimization logs, Delta history previews, and sample query timing results.

In [ ]:
from datetime import datetime, timezone
from time import perf_counter
from typing import Dict, List

GOLD_TABLES = ["dim_customer", "dim_account", "dim_product", "dim_branch", "dim_date", "fact_transaction"]
ENABLE_OPTIMIZE = True
ENABLE_VACUUM = False
VACUUM_RETENTION_HOURS = 168

def log_info(message: str) -> None:
    print(f"[INFO] {datetime.now(timezone.utc).isoformat()} | {message}")

def log_warning(message: str) -> None:
    print(f"[WARN] {datetime.now(timezone.utc).isoformat()} | {message}")

def safe_sql(sql_text: str):
    try:
        log_info(sql_text)
        return spark.sql(sql_text)
    except Exception as exc:
        log_warning(f"SQL skipped or failed: {sql_text}. Reason: {exc}")
        return None

## Inspect table health before optimization

In [ ]:
def collect_table_health(table_name: str) -> Dict[str, object]:
    row = {"table_name": table_name, "exists": spark.catalog.tableExists(table_name), "row_count": None, "num_files": None, "size_in_bytes": None, "format": None, "status": "UNKNOWN"}
    if not row["exists"]:
        row["status"] = "MISSING"
        return row

    row["row_count"] = spark.table(table_name).count()
    detail_df = safe_sql(f"DESCRIBE DETAIL {table_name}")
    if detail_df is not None:
        try:
            detail = detail_df.collect()[0].asDict()
            row["num_files"] = detail.get("numFiles")
            row["size_in_bytes"] = detail.get("sizeInBytes")
            row["format"] = detail.get("format")
        except Exception as exc:
            log_warning(f"Could not parse DESCRIBE DETAIL for {table_name}: {exc}")
    row["status"] = "READY"
    return row

before_health_df = spark.createDataFrame([collect_table_health(table_name) for table_name in GOLD_TABLES])
display(before_health_df.orderBy("table_name"))

## Run safe optimization

OPTIMIZE is useful when tables have many small files. This sample dataset is tiny, so the cell demonstrates the operational pattern more than the performance benefit.

In [ ]:
optimization_rows: List[Dict[str, object]] = []

for table_name in GOLD_TABLES:
    started = perf_counter()
    status = "SKIPPED"
    message = "Optimization disabled"

    if ENABLE_OPTIMIZE:
        result_df = safe_sql(f"OPTIMIZE {table_name}")
        status = "SUCCESS" if result_df is not None else "REVIEW"
        message = "OPTIMIZE executed" if result_df is not None else "OPTIMIZE not supported or failed in this environment"

    optimization_rows.append({"table_name": table_name, "operation": "OPTIMIZE", "status": status, "duration_seconds": round(perf_counter() - started, 3), "message": message})

optimization_df = spark.createDataFrame(optimization_rows)
display(optimization_df.orderBy("table_name"))

## Vacuum guidance

VACUUM removes old files that support time travel. It is disabled by default because retention decisions depend on recovery requirements, audit requirements, and downstream refresh schedules.

In [ ]:
vacuum_rows = []

for table_name in GOLD_TABLES:
    if ENABLE_VACUUM:
        result_df = safe_sql(f"VACUUM {table_name} RETAIN {VACUUM_RETENTION_HOURS} HOURS")
        status = "SUCCESS" if result_df is not None else "REVIEW"
        message = f"VACUUM attempted with {VACUUM_RETENTION_HOURS} hour retention"
    else:
        status = "SKIPPED"
        message = "Vacuum disabled. Review retention and recovery requirements before enabling."
    vacuum_rows.append({"table_name": table_name, "operation": "VACUUM", "status": status, "message": message})

display(spark.createDataFrame(vacuum_rows).orderBy("table_name"))

## Inspect table health and history after maintenance

In [ ]:
after_health_df = spark.createDataFrame([collect_table_health(table_name) for table_name in GOLD_TABLES])
display(after_health_df.orderBy("table_name"))

for table_name in GOLD_TABLES:
    history_df = safe_sql(f"DESCRIBE HISTORY {table_name} LIMIT 5")
    if history_df is not None:
        log_info(f"History for {table_name}")
        display(history_df)

## Query timing example

In [ ]:
query = """
    SELECT
        d.year_month,
        p.product_category,
        COUNT(DISTINCT f.transaction_id) AS transaction_count,
        SUM(f.absolute_transaction_amount) AS total_transaction_amount
    FROM fact_transaction f
    JOIN dim_date d ON f.date_key = d.date_key
    JOIN dim_product p ON f.product_key = p.product_key
    WHERE f.is_posted = true
    GROUP BY d.year_month, p.product_category
    ORDER BY d.year_month, p.product_category
"""

started = perf_counter()
timing_df = spark.sql(query)
result_count = timing_df.count()
duration_seconds = round(perf_counter() - started, 3)

log_info(f"Performance sample returned {result_count} rows in {duration_seconds} seconds")
display(timing_df)

## Next steps

Run 06_powerbi_ready_views.ipynb to validate Power BI-friendly reporting shapes, then use the SQL scripts in the sql folder to create persistent SQL Analytics Endpoint views.